<a href="https://colab.research.google.com/github/BhogathiChaitanyaKumarReddy/BhogathiChaitanyaKumarReddy/blob/main/DocGenius.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered PDF Q&A App with Streamlit and Gemini

This notebook sets up and runs an AI-powered PDF Q&A application using Streamlit and the Gemini API. It handles dependency installation, repository cloning, ngrok tunneling for public access, and secure API key management.

### 1. Install Dependencies

In [ ]:
import subprocess
import sys

# Added PyPDF2 to the list
packages = [
    'streamlit',
    'pyngrok',
    'google-generativeai',
    'PyPDF2'
]

for pkg in packages:
    try:
        __import__(pkg)
        print(f"{pkg} is already installed.")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"{pkg} installed successfully.")

streamlit is already installed.
pyngrok is already installed.
Installing google-generativeai...
google-generativeai installed successfully.


### 2. Clone the Project Repository

In [ ]:
import os
import shutil

repo_url = "https://github.com/Anmol-Dhiman/DocGenius-Revolutionizing-PDFs-with-AI.git"
# Clean up any failed previous attempts
if os.path.exists("/content/repo_temp"):
    shutil.rmtree("/content/repo_temp")

print(f"Cloning {repo_url}...")
!git clone {repo_url} /content/repo_temp

# Move files to /content/ so app.py is easy to find
if os.path.exists("/content/repo_temp"):
    for item in os.listdir("/content/repo_temp"):
        s = os.path.join("/content/repo_temp", item)
        d = os.path.join("/content/", item)
        if os.path.exists(d): continue
        shutil.move(s, d)
    print("Repository files moved to /content/")

os.chdir("/content/")
print(f"Current working directory: {os.getcwd()}")

Cloning https://github.com/Anmol-Dhiman/DocGenius-Revolutionizing-PDFs-with-AI.git...
Cloning into '/content/repo_temp'...
fatal: could not read Username for 'https://github.com': No such device or address
Current working directory: /content


### 3. Set Up ngrok for Public Access

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import getpass

ngrok.kill()

try:
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
    print("Found ngrok token in secrets.")
except:
    print("NGROK_AUTH_TOKEN not found in Secrets. Please enter it manually:")
    NGROK_AUTH_TOKEN = getpass.getpass()

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authentication token set successfully!")

NGROK_AUTH_TOKEN not found in Secrets. Please enter it manually:
··········
ngrok authentication token set successfully!


### 4. Configure Google API Key

In [87]:
import os
from google.colab import userdata
import google.generativeai as genai
import getpass

try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    print("Found Google API Key in secrets.")
except:
    print("GOOGLE_API_KEY not found in Secrets. Please enter it manually:")
    GOOGLE_API_KEY = getpass.getpass()

if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    app_file_path = "/content/app.py"

    definitive_app_content = '''
import os
import streamlit as st
import google.generativeai as genai
from PyPDF2 import PdfReader

# Configure Gemini API
if os.environ.get('GOOGLE_API_KEY'):
    genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))

def extract_text_from_pdf(file):
    reader = PdfReader(file)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\\n"
    return text

# Streamlit app UI
st.title("DocGenius: PDF Q&A with Gemini")

# File uploader
uploaded_file = st.file_uploader("Choose a PDF file", type="pdf")

# Text input for user prompt
user_input = st.text_input("Ask a question about the document:")

if uploaded_file is not None:
    st.success(f"File '{uploaded_file.name}' uploaded successfully!")

    if user_input:
        with st.spinner("Processing PDF content..."):
            # 1. Extract text from the PDF
            pdf_text = extract_text_from_pdf(uploaded_file)

            # 2. Use the verified model name
            model = genai.GenerativeModel("gemini-2.5-flash")

            # 3. Construct the prompt with context (truncating to fit within typical limits)
            prompt = f"""
            Answer the question using only the PDF content provided below.\\n\\n
            PDF Content:\\n{pdf_text[:30000]}\\n\\n
            Question: {user_input}
            """

            response = model.generate_content(prompt)

            st.write("### Response from Gemini AI:")
            st.write(response.text)
else:
    st.info("Please upload a PDF file to begin.")
'''

    with open(app_file_path, "w") as f:
        f.write(definitive_app_content)
    print("app.py updated with gemini-2.5-flash and full PDF text extraction. Restart Streamlit now.")

GOOGLE_API_KEY not found in Secrets. Please enter it manually:
··········
app.py updated with gemini-2.5-flash and full PDF text extraction. Restart Streamlit now.


### 5. Run the Streamlit Application

In [ ]:
import subprocess
import threading
import time
import sys
import os
from pyngrok import ngrok

# Configuration
PORT = 8502

def run_streamlit():
    env_vars = os.environ.copy()
    if 'GOOGLE_API_KEY' in os.environ:
        env_vars['GOOGLE_API_KEY'] = os.environ['GOOGLE_API_KEY']

    with open('/content/streamlit.log', 'w') as f:
        cmd = [
            sys.executable, "-m", "streamlit", "run",
            "app.py",
            "--server.port", str(PORT),
            "--server.headless", "true"
        ]
        subprocess.Popen(cmd, env=env_vars, stdout=f, stderr=f)

print(f"Starting Streamlit server on port {PORT}...")
threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(10)

try:
    # Connect ngrok to the new port
    public_url = ngrok.connect(PORT)
    url_str = public_url.public_url if hasattr(public_url, 'public_url') else str(public_url)
    print(f"\nSuccess! Access your project here: {url_str}")
except Exception as e:
    print(f"\nTunnel error: {e}")

print("\n--- Streamlit Logs ---")
if os.path.exists('/content/streamlit.log'):
    with open('/content/streamlit.log', 'r') as f:
        print(f.read())
else:
    print("Log file not found.")

Starting Streamlit server on port 8502...

Success! Access your project here: https://comment-preacher-banter.ngrok-free.dev

--- Streamlit Logs ---


2026-06-06 17:56:04.836 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://8.229.30.166:8502




### 6. Restart Streamlit Server
Run the cell below to apply the updates to `app.py`.

In [88]:
!pkill -f streamlit
print("Restarting Streamlit with updated app.py...")
!streamlit run app.py > streamlit.log 2>&1 &

Restarting Streamlit with updated app.py...


### 7. Reset and Reconnect Tunnel
Use this cell if you see an ngrok error or if the app stops responding.

In [89]:
import os
import time
from pyngrok import ngrok

# 1. Kill old processes
!pkill -f streamlit
!pkill -f ngrok
ngrok.kill()

# 2. Restart Streamlit
print("Starting Streamlit...")
!streamlit run app.py --server.port 8502 --server.headless true > streamlit.log 2>&1 &
time.sleep(15)

# 3. Check logs for success
with open('streamlit.log', 'r') as f:
    log_content = f.read()
    print("--- Streamlit Log Snippet ---")
    print('\n'.join(log_content.splitlines()[-5:]))

# 4. Create new tunnel
try:
    public_url = ngrok.connect(8502)
    print(f"\nNew Public URL: {public_url.public_url}")
except Exception as e:
    print(f"Tunnel Error: {e}")

Starting Streamlit...
--- Streamlit Log Snippet ---

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://8.229.30.166:8502


New Public URL: https://comment-preacher-banter.ngrok-free.dev


In [ ]:
import os
# Check if app.py exists and show its first few lines to verify configuration
if os.path.exists('app.py'):
    print('--- app.py content start ---')
    with open('app.py', 'r') as f:
        print(f.read())
    print('--- app.py content end ---')
else:
    print('app.py NOT found in /content/')

# Check for running streamlit processes
print('\nChecking for running streamlit processes:')
!ps aux | grep streamlit

--- app.py content start ---

import os
import streamlit as st
import google.generativeai as genai

# Ensure Gemini API is configured with the GOOGLE_API_KEY from the environment
# This addresses potential issues with subprocess not inheriting env vars correctly.
# Made unconditional if key exists to prevent issues with genai._configured_api_key state.
if os.environ.get('GOOGLE_API_KEY'):
    genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))

# Streamlit app
st.title("Gemini AI Content Generator")

# Text input for user prompt (with form submission on Enter)
user_input = st.text_input("Enter a prompt:")

# Check if user input is provided
if user_input:
    # Use the Gemini model to generate content
    model = genai.GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(user_input)

    # Display the generated response
    st.write("Response from Gemini AI:")
    st.write(response.text)
else:
    st.write("Please enter a prompt to generate content.")

--- app

In [ ]:
# 1. Kill any existing Streamlit and ngrok processes to start fresh
!pkill streamlit
!pkill ngrok

# 2. Clear the ngrok cache to prevent tunnel reuse issues
from pyngrok import ngrok
ngrok.kill()

print("Background processes cleared. Please re-run the 'Set Up ngrok' and 'Run the Streamlit Application' cells.")

Background processes cleared. Please re-run the 'Set Up ngrok' and 'Run the Streamlit Application' cells.


### 6. Upgrade `google-generativeai` SDK

In [83]:
import subprocess
import sys

print("Upgrading google-generativeai...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "google-generativeai"])
print("google-generativeai upgraded successfully. Please restart the runtime now (Runtime > Restart runtime...).")

Upgrading google-generativeai...
google-generativeai upgraded successfully. Please restart the runtime now (Runtime > Restart runtime...).


In [85]:
import google.generativeai as genai
import os

# Retrieve the API key from the environment variable set in Step 4
api_key = os.environ.get('GOOGLE_API_KEY')

if not api_key:
    print('API Key not found in environment. Please re-run the "Configure Google API Key" cell (Step 4).')
else:
    genai.configure(api_key=api_key)
    print('Available models:')
    try:
        for model in genai.list_models():
            if 'generateContent' in model.supported_generation_methods:
                print(model.name)
    except Exception as e:
        print(f'Error listing models: {e}')

Available models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2